# Miscellaneous

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 02.07.2025
- Date of Last Modification: 02.07.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

## Load Tokenized Data

In [59]:
from datasets import load_from_disk

data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10.dataset' # '/fsx-shared/s3/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10.dataset' # hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)
dataset

Dataset({
    features: ['cell_degrees', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value_token', 'gene_panel_value_token', 'assay_value_token', 'species_value_token', 'tissue_value_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens', 'cls_tokens', 'cell_total_counts', 'cell_n_probed_genes'],
    num_rows: 80172337
})

In [ ]:
from datasets import load_from_disk
data_path = '/fsx-shared/s3/hst_corpus_80m/hst_corpus_80m_1k_subset.dataset' # '/fsx-shared/s3/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10.dataset' # hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)
dataset

In [60]:
subset = dataset.select(range(1000))

# Save the subset to disk
subset_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_1k_subset.dataset'
subset.save_to_disk(subset_path)

Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 2690.41 examples/s]


In [45]:
item = dataset[0]
item

In [1]:
tokenizer_folder_path = '/nfs/team361/sb75/DATASETS/tokenizer'
tokenized_data_str = 'hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10' # 'hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10'
n_samples = 30

In [30]:
import pickle

with open(f'{tokenizer_folder_path}/{tokenized_data_str}_precomputed_epoch_split_{n_samples}.pkl', 'rb') as f:
    split = pickle.load(f)

## Loop and Combine Raw Data

In [58]:
import os
import scanpy as sc

for subdir, _, files in os.walk('/nfs/team361/sb75/DATASETS/silver/xhb1002-AT9-6b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(file_path)
            print(adata.obs['cell_id'].values[0])
            #print(adata.obs['sample'].unique())
            print(adata.uns['batch'])
            print("================")

/nfs/team361/sb75/DATASETS/silver/xhb1002-AT9-6b_1p/AT9-BRA-4-FO-2-S38__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad
1002_batch3_0
batch3
/nfs/team361/sb75/DATASETS/silver/xhb1002-AT9-6b_1p/AT9-BRA-4-FO-4-S45-i__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad
1002_batch15_0
batch15



KeyboardInterrupt



In [ ]:
# Load hold out skin samples and concatenate in one adata object
adata_list = []

for subdir, _, files in os.walk('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(adata.X)
            adata.X = adata.X.astype(int64)
            print(adata.X)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
max_nonzero_counts = 0  # Initialize a variable to store the maximum count
neighbor_max_nonzero_counts = 0

# Iterate through all .h5ad files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.h5ad'):
        file_path = os.path.join(folder_path, filename)
        
        # Load the .h5ad file
        adata = sc.read_h5ad(file_path)

        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                delaunay=True,
                                set_diag=True)

        adata.layers['X_neighborhood'] = (
                adata.obsp['spatial_connectivities'].T @ adata.X)
        
        # Calculate nonzero counts per cell
        nonzero_counts = (adata.X > 0).sum(axis=1)
        neighbor_nonzero_counts = (adata.layers['X_neighborhood'] > 0).sum(axis=1)
        
        # Get the maximum nonzero count in the current file
        file_max_nonzero = nonzero_counts.max()
        file_max_neighbor_nonzero = neighbor_nonzero_counts.max()
        
        # Update the overall maximum if the current file has a higher value
        if file_max_nonzero > max_nonzero_counts:
            max_nonzero_counts = file_max_nonzero

        if file_max_neighbor_nonzero > neighbor_max_nonzero_counts:
            neighbor_max_nonzero_counts = file_max_neighbor_nonzero

## Load Token Dictionary

In [ ]:
import pickle

# Path to your pickle file
pickle_file_path = '/fsx-shared/s3/tokenizer/token_dictionary_homo_sapiens.pkl' # '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_10m/token_dictionary.pkl'

# Load the pickle file
with open(pickle_file_path, 'rb') as file:
    token_dict = pickle.load(file)

# Now `token_dict` is a Python dictionary
print(token_dict)

In [ ]:
spv_count = sum(1 for key in token_dict if "spv" in key)
spv_count

## Data Analysis

In [ ]:
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='niche_name')

In [ ]:
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.X = sc.pp.log1p(scales_counts["X"], copy=True)
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color='niche_name')

In [ ]:
sc.pl.spatial(adata,
              color='niche_name',
              spot_size=30)

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    inplace=True
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data=adata.obs, x='n_genes_by_counts', bins=100)
plt.xlabel('Number of expressed genes per cell')
plt.ylabel('Number of cells')
plt.show()

## Skin Sample Mapping

In [54]:
import scanpy as sc

adata_test = sc.read_h5ad('/nfs/team361/sb75/DATASETS/bronze/xhs1000-39b_1p/adata_5k_xenium_skin.h5ad')

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [57]:
batch_mapping_dict = {value: idx for idx, value in enumerate(adata_test.obs['info_id'].unique().tolist())}
batch_mapping_dict

{'BK27_Week 12': 0,
 'BK27_Day 14': 1,
 'BK21_Lesional Baseline': 2,
 'BK20_Lesional Baseline': 3,
 'BK18_Non-lesional Baseline': 4,
 'BK21_Day 3': 5,
 'BK18_Week 12': 6,
 'BK20_Day 14': 7,
 'BK20_Non-lesional Baseline': 8,
 'BK27_Non-lesional Baseline': 9,
 'BK21_Day 14': 10,
 'BK18_Lesional Baseline': 11,
 'BK18_Day 3': 12,
 'BK27_Day 3': 13,
 'BK21_Non-lesional Baseline': 14,
 'BK20_Week 12': 15,
 'BK21_Week 12': 16,
 'BK20_Day 3': 17,
 'BK18_Day 14': 18,
 'BK39_Lesional Baseline': 19,
 'BK30_Non-lesional Baseline': 20,
 'BK39_Non-lesional Baseline': 21,
 'BK39_Week 12': 22,
 'BK25_Day 3': 23,
 'BK25_Day 14': 24,
 'BK25_Non-lesional Baseline': 25,
 'BK30_Day 3': 26,
 'BK30_Day 14': 27,
 'BK25_Lesional Baseline': 28,
 'BK25_Week 12': 29,
 'BK24_Lesional Baseline': 30,
 'BK24_Day 14': 31,
 'BK30_Lesional Baseline': 32,
 'BK24_Non-lesional Baseline': 33,
 'BK39_Day 14': 34,
 'BK24_Day 3': 35,
 'BK30_Week 12': 36,
 'BK24_Week 12': 37,
 'BK39_Day 3': 38}

In [ ]:
adata_test.obs[['sample_id', 'Site_status']].drop_duplicates().values.tolist()

## Zip Directories

## Investigate Pickle File Differences

In [ ]:
# Pickle the list
with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    train = pickle.load(f)

with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    test = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    new_train = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    new_test = pickle.load(f)

print(len(train))
print(len(test))
print(len(new_train))
print(len(new_test))

train_matches = set(train).intersection(new_train)
print(len(train_matches))
test_matches = set(test).intersection(new_test)
print(len(test_matches))

train_differences = []
for i in range(len(train)):
    if train[i] != new_train[i]:
        train_differences.append((i, train[i], new_train[i]))
print(train_differences)
print(len(differences))

test_differences = []
for i in range(len(test)):
    if test[i] != new_test[i]:
        test_differences.append((i, test[i], new_test[i]))
print(test_differences[-10:])

## Harmonize mmb0-1b_smb1-1b_2p

## Gene Name to Ensembl ID

In [ ]:
with open('/lustre/scratch126/cellgen/team361/DATASETS/genes/homo_sapiens_gene_name_to_ensembl_id_dict.pkl', 'rb') as f:
    gene_name_to_ensembl_id_dict = pickle.load(f)
    
print(gene_name_to_ensembl_id_dict)

## Check Infra Resource Availability

In [ ]:
!free -h

## Check Perturbed Cell IDs

In [ ]:
# Define color map for boolean values
bool_colors = {True: "orange", False: "lightgrey"}

# Ensure it's treated as categorical (not continuous)
adata.obs["pert_cell_id"] = adata.obs["pert_cell_id"].astype("category")

# Visualize
sc.pl.spatial(
    adata,
    color="pert_cell_id",
    spot_size=spot_size,
    palette=bool_colors
)

## Test Cell Dataset

In [1]:
from datasets import load_from_disk

data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10.dataset' # '/fsx-shared/s3/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10.dataset' # hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)
dataset

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 80172337
})

In [9]:
columns = list(dataset.features.keys())
columns.remove("cell_id")
dataset.set_format(
    type="torch",
    columns=columns,
    output_all_columns=True)
dataset # 700,000 cells

Dataset({
    features: ['rel_x_coord', 'rel_y_coord', 'cell_id', 'batch_token', 'gene_panel_token', 'assay_token', 'species_token', 'tissue_token', 'batch_value', 'gene_panel_value', 'assay_value', 'species_value', 'tissue_value', 'gene_tokens', 'gene_expr', 'seg_tokens', 'n_nonzero_tokens'],
    num_rows: 80172337
})

In [27]:
import pickle

token_dict_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

with open(token_dict_folder_path, 'rb') as file:
    token_dict = pickle.load(file)
vocab_size = len(token_dict)

seq_len_cell = 256
seq_len_neighborhood = 256
tokenizer_type = 'cell_graph'
gt_type = 'combined'
cell_pos_enc = 'coord'
special_tokens = []
sampling_strategy = None
n_nonzero_tokens = None
sep_gene_tokens_neb = False

In [28]:
from nichejepa.datasets.cell_datasets import init_cell_dataset

cell_dataset = init_cell_dataset(
    dataset=dataset,
    vocab_size=vocab_size,
    seq_len_cell=seq_len_cell,
    seq_len_neighborhood=seq_len_neighborhood,
    tokenizer_type=tokenizer_type,
    gt_type=gt_type,
    cell_pos_enc=cell_pos_enc,
    special_tokens=special_tokens,
    sampling_strategy=sampling_strategy,
    n_nonzero_tokens_list=n_nonzero_tokens,
    include_cell_id=False,
    sep_gene_tokens_neb=sep_gene_tokens_neb)

In [29]:
cell_data = cell_dataset[10]

In [31]:
cell_data['tokens'][:257]

tensor([ 1415,  9167, 10413, 12154,  3295,  6247,   655,  9203,  9067, 16997,
         9440, 16259, 12722,  4815,  8847,  7230, 10669, 13089, 12985,   853,
         9172,  3484,  6786, 13016,  1550,  5680,  4806,  1026,  4830, 16420,
         3622, 10824,  6867,  6973, 16214, 15480,  9523, 16382, 14518, 12282,
         5385, 11801, 14036,   217,  7860, 11954,  2859,  3816,  9862,  4566,
         5432,  8117,  3944,  7084,   498,  9007,  8203,  7001,  3524,   695,
        14747, 12991,  3307,  2939, 14681,  6618,  6473,  5395, 16406, 15208,
         7389,  7051,  6877,  8188,  8192,  4543,  5036,  2161,  8155, 14180,
         5158,  4563,  4209, 13608,  4167,  2308,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0, 

In [36]:
cell_data['segments'][:257]

tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2])

In [30]:
cell_data

{'tokens': tensor([ 1415,  9167, 10413, 12154,  3295,  6247,   655,  9203,  9067, 16997,
          9440, 16259, 12722,  4815,  8847,  7230, 10669, 13089, 12985,   853,
          9172,  3484,  6786, 13016,  1550,  5680,  4806,  1026,  4830, 16420,
          3622, 10824,  6867,  6973, 16214, 15480,  9523, 16382, 14518, 12282,
          5385, 11801, 14036,   217,  7860, 11954,  2859,  3816,  9862,  4566,
          5432,  8117,  3944,  7084,   498,  9007,  8203,  7001,  3524,   695,
         14747, 12991,  3307,  2939, 14681,  6618,  6473,  5395, 16406, 15208,
          7389,  7051,  6877,  8188,  8192,  4543,  5036,  2161,  8155, 14180,
          5158,  4563,  4209, 13608,  4167,  2308,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0

In [46]:
dataset_ids = [
'xhb23-1b_1p',
'xhb24-2b_1p',
'xhb25-2b_1p',
'xhb26-2b_1p',
'xhb27-1b_1p',
'xhl28-1b_1p',
'xhl29-1b_1p',
'xhp30-1b_1p',
'xhs31-2b_1p',
'xhh32-1b_1p',
'xhln33-1b_1p',
'xhp34-1b_1p',
'xhp35-1b_1p',
'xhs36-2b_1p',
'xhb37-3b_1p',
'xhc38-4b_1p',
'xhk39-2b_1p',
'xhl40-2b_2p',
'xhl41-2b_1p',
'xhb42-3b_1p',
'xht43-2b_1p',
'xhsc44-6b_1p',
'xhc45-1b_1p',
'xhl46-1b_1p',
'xhp48-1b_1p',
'xhs49-1b_1p',
'xho50-1b_1p',
'xhc51-1b_1p',
'xhd52-1b_1p',
'xhl53-1b_1p',
'xho54-1b_1p',
'xhc55-3b_1p',
'xhs62-1b_1p',
'xhb76-16b_1p',
'xhk79-4b_1p',
'xhs1000-39b_1p',
'xhb1002-AT3-1b_1p',
'xhb1002-AT4-3b_1p',
'xhb1002-AT5-1b_1p',
'xhb1002-AT9-6b_1p',
'xhb1002-AT10-7b_1p',
'xhb1002-AT12-1b_1p',
'xhb1002-AT13-2b_1p',
'xhb1002-AT14-1b_1p',
'xhb1002-AT15-10b_1p',
'xhb1002-AT16-15b_1p',
'xhb1002-AT17-2b_1p',
'xhb1002-AT18-2b_1p',
'xhb1002-AT19-1b_1p',
'xhb1002-AT20-2b_1p',
'xhb1002-AT21-2b_1p',
'xhb1002-AT23-9b_1p',
'xhb1002-AT101-6b_1p',
'xhb1003-1b_1p',
'xhln1004-1b_1p',
'xhl1005-1b_1p',
'mhb14-1b_1p',
'mhc15-2b_1p',
'mhl16-2b_1p',
'mhl17-2b_1p',
'mhm18-2b_1p',
'mho19-4b_1p',
'mhp20-2b_1p',
'mhu21-3b_1p',
'mhl63-10b_1p',
'chb56-1b_1p',
'chl57-2b_1p',
'chl59-8b_1p',
'chc61-9b_1p',
'chr78-11b_1p',
'chp60-1b_1p',
'mhh22-3b_1p',
'shp75-4b_1p',
'xhb1002-AT6-1-1b_1p',
'xhb1002-AT6-2-1b_1p',
'xhb1002-AT6-3-1b_1p',
'xhb1002-AT6-4-1b_1p',
'xho47-S1-1b_1p',
'xho47-S2-1b_1p',
'xhsc1006-2b_1p',
'xhc1001-1b_1p',
'mhl80-8b_1p',
'xhb81-1b_1p',
'xhl82-2b_1p',
'mhb83-1b_1p',
'xhk1007-CV1-41b_1p',
'xhk1007-CV2-41b_1p',
'xhk1007-CV3-41b_1p',
'xhk1007-CV4-41b_1p',
'xhk1007-CV5-41b_1p',
'xhk1007-CV6-41b_1p',
'xhk1007-CV7-41b_1p',
'xhk1007-CV8-41b_1p',
'xhk1007-CV9-41b_1p',
'xhe1008-1-20b_1p',
'xhe1008-2-20b_1p',
'xhe1008-3-20b_1p',
'xhe1008-4-20b_1p',
'xhe1008-5-20b_1p',
'xhe1008-6-20b_1p',
'xhe1008-7-20b_1p',
'xhe1008-8-20b_1p',
'xhe1008-9-20b_1p',
'xhe1008-10-20b_1p',
'xhs1009-17b_1p',
'xhs1010-27b_1p',
'xhs1011-22b_1p',
'xho1012-AU14-6b_1p',
'xho1012-AU16-7b_1p',
'xho1012-AU18-6b_1p',
'xho1012-AU20-6b_1p',
'xho1012-AU21-7b_1p',
'xho1012-AU22-6b_1p',
'xho1012-AU30-6b_1p',
'xho1012-AUrv15-6b_1p',
'xho1012-Hrv66-2b_1p',
'xho1012-Hrv69-4b_1p',
'xho1012-Hrv162-2b_1p',
'xho1012-Hrv182-4b_1p',
'xho1012-M23-4b_1p',
'xhl1013-4b_1p']

In [47]:
config_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/config/bronze_to_silver'
silver_folder_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver'

import os
import yaml
import scanpy as sc

yaml_data = []

# Loop through the directory
for config_file_name in os.listdir(config_folder_path):
    
    if (config_file_name.endswith(".yaml") or config_file_name.endswith(".yml")) and (config_file_name.split('.')[0] in dataset_ids):
        config_file_path = os.path.join(config_folder_path, config_file_name)
        with open(config_file_path, 'r') as f:
            try:
                data = yaml.safe_load(f)
                yaml_data.append(data)
            except yaml.YAMLError as exc:
                print(f"Error parsing {filename}: {exc}")

        try:
    
            if 'condition' in data['harmonization']['uns'].keys():
                condition = data['harmonization']['uns']['condition']
    
                adata_silver_folder_path = f'{silver_folder_path}/{config_file_name.split(".")[0]}'
                for subdir, _, files in os.walk(adata_silver_folder_path):
                    for file_idx, file in enumerate(files):
                        if file.endswith('.h5ad'):
                            adata_file_path = os.path.join(subdir, file)
                            
                            adata = sc.read_h5ad(adata_file_path)
                            adata.uns['condition'] = condition
                            adata.write(adata_file_path)
                
            else:
    
                adata_silver_folder_path = f'{silver_folder_path}/{config_file_name.split(".")[0]}'
                for subdir, _, files in os.walk(adata_silver_folder_path):
                    for file_idx, file in enumerate(files):
                        if file.endswith('.h5ad'):
                            adata_file_path = os.path.join(subdir, file)
    
                            if file.split(".")[0] in data['harmonization']['uns']['sample_condition_map'].keys():
                                condition = data['harmonization']['uns']['sample_condition_map'][file.split(".")[0]]
                                adata = sc.read_h5ad(adata_file_path)
                                adata.uns['condition'] = condition
                                adata.write(adata_file_path)
                            else:
                                adata = sc.read_h5ad(adata_file_path)
                                sample = adata.obs[data['harmonization']['batch']['batch_key']].values.tolist()[0]
                                condition = data['harmonization']['uns']['sample_condition_map'][sample]
                                adata.uns['condition'] = condition
                                adata.write(adata_file_path)

            print('=====')
            print("SUCCESS")
            print(file)
            print(adata_silver_folder_path)
            print(condition)
            print('=====')

        except Exception as e:
            print('=====')
            print(f"An error occurred: {e}")
            print("FAILURE")
            print(file)
            print(adata_silver_folder_path)
            print(condition)
            print('=====')
            

# Now `yaml_data` is a list of all the loaded YAML contents
print(f"Loaded {len(yaml_data)} YAML files.")

=====
SUCCESS
Xenium_V1_FFPE_Human_Brain_Alzheimers_With_Addon_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb42-3b_1p
alzheimer_disease
=====
=====
SUCCESS
MERSCOPE_Human_Colon_Cancer_Patient2.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhc15-2b_1p
adenocarcinoma
=====
=====
SUCCESS
Xenium_V1_FFPE_Human_Breast_IDC_With_Addon_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb25-2b_1p
invasive_ductal_carcinoma
=====


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


=====
SUCCESS
NSCLC_healthy_lymph_node.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhln1004-1b_1p
healthy
=====
=====
SUCCESS
Xenium_V1_hLymphNode_nondiseased_section_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhln33-1b_1p
healthy
=====
=====
SUCCESS
Xenium_V1_hColon_Cancer_Base_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhc38-4b_1p
adenocarcinoma
=====
=====
SUCCESS
MERSCOPE_Human_Breast_Cancer_Patient1.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhb14-1b_1p
invasive_ductal_carcinoma
=====
=====
SUCCESS
Xenium_V1_hSkin_nondiseased_section_2_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhs36-2b_1p
healthy
=====
=====
SUCCESS
Pancreas-CosMx-WTx-FlatFiles_manual_extraction.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/chp60-1b_1p
healthy
=====
=====
SUCCESS
NicheCompass_hs_XeniumAdultColon_AM_07112024_132853_raw.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhc10

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-p

=====
SUCCESS
adata_batch6.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhs1000-39b_1p
eczema
=====
=====
SUCCESS
adata_batch2.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhh22-3b_1p
healthy
=====
=====
SUCCESS
Xenium_Prime_Ovarian_Cancer_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xho50-1b_1p
serous_carcinoma
=====
=====
SUCCESS
MERSCOPE_Human_Lung_Cancer_Patient1.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhl17-2b_1p
nonsmall_cell_lung_carcinoma
=====
=====
SUCCESS
Xenium_Preview_Human_Non_diseased_Lung_With_Add_on_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhl41-2b_1p
healthy
=====
=====
SUCCESS
Xenium_V1_Human_Colorectal_Cancer_Addon_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhc51-1b_1p
adenocarcinoma
=====
=====
SUCCESS
Xenium_V1_FFPE_Human_Breast_IDC_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb26-2b_1p
invasive_ductal_carcinoma
=====
==

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


=====
SUCCESS
NSCLC_PT.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhl1005-1b_1p
nonsmall_cell_lung_carcinoma
=====
=====
SUCCESS
Xenium_V1_Human_Ductal_Adenocarcinoma_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhd52-1b_1p
ductal_adenocarcinoma
=====
=====
SUCCESS
Xenium_V1_hPancreas_nondiseased_section_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhp35-1b_1p
healthy
=====


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


=====
SUCCESS
breast_Xenium_no_exp_qv.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb1003-1b_1p
invasive_ductal_carcinoma
=====
=====
SUCCESS
122_region_1_cell_.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhl63-10b_1p
hepatocellular_carcinoma
=====
=====
SUCCESS
Xenium_Prime_Human_Prostate_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhp48-1b_1p
adenocarcinoma
=====
=====
SUCCESS
Xenium_V1_Human_Brain_GBM_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb23-1b_1p
glioblastoma
=====
=====
SUCCESS
Xenium_V1_hLung_cancer_section_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhl29-1b_1p
adenocarcinoma
=====


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


=====
SUCCESS
adata_batch6.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhs1009-17b_1p
eczema
=====
=====
SUCCESS
MERSCOPE_Human_Ovarian_Cancer_Patient1.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mho19-4b_1p
serous_carcinoma
=====
=====
SUCCESS
Xenium_V1_human_Pancreas_FFPE_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhp34-1b_1p
healthy
=====
=====
SUCCESS
Xenium_V1_FFPE_Human_Breast_IDC_Big_1_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb24-2b_1p
invasive_ductal_carcinoma
=====
=====
SUCCESS
5730P.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb76-16b_1p
healthy
=====
=====
SUCCESS
Xenium_V1_hBoneMarrow_nondiseased_section_outs.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/xhb37-3b_1p
healthy
=====
=====
SUCCESS
MERSCOPE_Human_Liver_Cancer_Patient1.h5ad
/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/mhl16-2b_1p
hepatocellular_carcinoma
=====
=====
SUCCESS
Xeniumranger_V1_hSkin_Mela

In [48]:
for folder_name in [
'xhb1002-AT3-1b_1p',
'xhb1002-AT4-3b_1p',
'xhb1002-AT5-1b_1p',
'xhb1002-AT9-6b_1p',
'xhb1002-AT10-7b_1p',
'xhb1002-AT12-1b_1p',
'xhb1002-AT13-2b_1p',
'xhb1002-AT14-1b_1p',
'xhb1002-AT15-10b_1p',
'xhb1002-AT16-15b_1p',
'xhb1002-AT17-2b_1p',
'xhb1002-AT18-2b_1p',
'xhb1002-AT19-1b_1p',
'xhb1002-AT20-2b_1p',
'xhb1002-AT21-2b_1p',
'xhb1002-AT23-9b_1p',
'xhb1002-AT101-6b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)
                adata.uns['condition'] = 'glioblastoma'
                adata.write(adata_file_path)

In [49]:
for folder_name in [
'xhb1002-AT6-1-1b_1p',
'xhb1002-AT6-2-1b_1p',
'xhb1002-AT6-3-1b_1p',
'xhb1002-AT6-4-1b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)
                adata.uns['condition'] = 'glioblastoma'
                adata.write(adata_file_path)

In [50]:
for folder_name in [
'xhe1008-1-20b_1p',
'xhe1008-2-20b_1p',
'xhe1008-3-20b_1p',
'xhe1008-4-20b_1p',
'xhe1008-5-20b_1p',
'xhe1008-6-20b_1p',
'xhe1008-7-20b_1p',
'xhe1008-8-20b_1p',
'xhe1008-9-20b_1p',
'xhe1008-10-20b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)
                adata.uns['condition'] = 'healthy'
                adata.write(adata_file_path)

In [51]:
for folder_name in [
'xho47-S1-1b_1p',
'xho47-S2-1b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)
                adata.uns['condition'] = 'adenocarcinoma'
                adata.write(adata_file_path)

In [52]:
for folder_name in [
'xho1012-AU14-6b_1p',
'xho1012-AU16-7b_1p',
'xho1012-AU18-6b_1p',
'xho1012-AU20-6b_1p',
'xho1012-AU21-7b_1p',
'xho1012-AU22-6b_1p',
'xho1012-AU30-6b_1p',
'xho1012-AUrv15-6b_1p',
'xho1012-Hrv66-2b_1p',
'xho1012-Hrv69-4b_1p',
'xho1012-Hrv162-2b_1p',
'xho1012-Hrv182-4b_1p',
'xho1012-M23-4b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)
                adata.uns['condition'] = 'healthy'
                adata.write(adata_file_path)

In [53]:
sample_condition_map = {
  'CV1-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV1-KID-0-FT-1': 'renal_cell_carcinoma',
  'CV1-KID-0-FT-2': 'renal_cell_carcinoma',
  'CV1-KID-0-FT-3': 'renal_cell_carcinoma',
  'CV2-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV2-KID-0-FO-2': 'renal_cell_carcinoma',
  'CV2-KID-0-FT-1-s2': 'renal_cell_carcinoma',
  'CV2-KID-0-FT-2': 'renal_cell_carcinoma',
  'CV3-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV3-KID-0-FT-2-s3': 'renal_cell_carcinoma',
  'CV3-KID-0-FT-3-s3': 'renal_cell_carcinoma',
  'CV3-KID-0-FT-4-s3': 'renal_cell_carcinoma',
  'CV3-KID-0-FT-5': 'renal_cell_carcinoma',
  'CV4-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV4-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV4-KID-0-FT-2-s3': 'renal_cell_carcinoma',
  'CV4-KID-0-FT-3-S3': 'renal_cell_carcinoma',
  'CV4-KID-0-FT-4-S3': 'renal_cell_carcinoma',
  'CV5-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV5-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV5-KID-0-FT-2-s3': 'renal_cell_carcinoma',
  'CV5-KID-0-FT-4-S3': 'renal_cell_carcinoma',
  'CV6-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV6-KID-0-FT-1': 'renal_cell_carcinoma',
  'CV6-KID-0-FT-2-s3': 'renal_cell_carcinoma',
  'CV6-KID-0-FT-3-s3': 'renal_cell_carcinoma',
  'CV6-KID-0-FT-4-S3': 'renal_cell_carcinoma',
  'CV6-KID-0-FT-5-S3': 'healthy',
  'CV7-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV7-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV7-KID-0-FT-2-s3': 'renal_cell_carcinoma',
  'CV7-KID-0-FT-3-S3': 'renal_cell_carcinoma',
  'CV7-KID-0-FT-4-S3': 'renal_cell_carcinoma',
  'CV8-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV8-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV9-KID-0-FO-1': 'renal_cell_carcinoma',
  'CV9-KID-0-FT-1-s3': 'renal_cell_carcinoma',
  'CV9-KID-0-FT-2-S3': 'renal_cell_carcinoma',
  'CV9-KID-0-FT-3-S3': 'renal_cell_carcinoma',
  'CV9-KID-0-FT-4-S3': 'renal_cell_carcinoma',
  'CV9-KID-0-FT-5-S4': 'healthy',
}

for folder_name in [
'xhk1007-CV1-41b_1p',
'xhk1007-CV2-41b_1p',
'xhk1007-CV3-41b_1p',
'xhk1007-CV4-41b_1p',
'xhk1007-CV5-41b_1p',
'xhk1007-CV6-41b_1p',
'xhk1007-CV7-41b_1p',
'xhk1007-CV8-41b_1p',
'xhk1007-CV9-41b_1p']:
    adata_silver_folder_path = f'{silver_folder_path}/{folder_name}'
    for subdir, _, files in os.walk(adata_silver_folder_path):
        for file_idx, file in enumerate(files):
            if file.endswith('.h5ad'):
                adata_file_path = os.path.join(subdir, file)
                adata = sc.read_h5ad(adata_file_path)

                sample = adata.obs['batch'].values.tolist()[0]
                condition = sample_condition_map[sample]
                adata.uns['condition'] = condition

                adata.write(adata_file_path)

In [ ]:
filename.split('.')[0]

In [ ]:
import os

# Walk through the load folder path and read files
if dataset_ids:
    print(f'Loading datasets: {dataset_ids}.')
    for subdir, _, files in os.walk(load_folder_path):
        if any(dataset_id in subdir.split('/')[-1].split('-')[0] for dataset_id in dataset_ids):
            print(f'Loading AnnData objects from {subdir}.')
            for file_idx, file in enumerate(files):
                if file.endswith('.h5ad'):
                    file_path = os.path.join(subdir, file)
                    adata = sc.read_h5ad(file_path)

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/nfs/team361/sb75/DATASETS/bronze/mhl80-8b_1p/adata_healthy_diseased_merfish.h5ad')

In [ ]:
adata.obs[['batch', 'condition']].drop_duplicates().values.tolist()

In [ ]:
adata = 'h5ad'

In [ ]:
from datasets import load_from_disk

data_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10.dataset'
dataset = load_from_disk(data_path)

In [ ]:
dataset

In [ ]:
item = dataset[0]

In [ ]:
item['rel_x_coord']

In [ ]:
rel_x_coord = dataset['rel_x_coord']

In [ ]:
import torch
rel_x_coord = torch.tensor(rel_x_coord)

In [ ]:
def repeat_feature_list(xs, repeat=256):
    return [x for x in xs for _ in range(repeat)]

In [ ]:
def repeat_column(batch):
    return {
        'rel_x_coord': [repeat_feature_list(x, repeat=256) for x in batch['rel_x_coord']]
    }

In [ ]:
dataset = dataset.map(
    repeat_column,
    batched=True,
    num_proc=12,  # Adjust based on your CPU cores
    load_from_cache_file=False
)

In [ ]:
def repeat_column(batch):
    return {
        'rel_y_coord': [repeat_feature_list(x, repeat=256) for x in batch['rel_y_coord']]
    }

In [ ]:
dataset = dataset.map(
    repeat_column,
    batched=True,
    num_proc=12,  # Adjust based on your CPU cores
    load_from_cache_file=False
)

In [ ]:
output_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10_updated.dataset'

In [ ]:
dataset.save_to_disk(output_path, num_shards=32)

In [ ]:
import torch

def repeat_rel_x_coord_batch(batch):
    return {'rel_x_coord': [x.repeat_interleave(256, dim=0) for x in batch['rel_x_coord']]}

dataset = dataset.map(
    repeat_rel_x_coord_batch,
    batched=True,
    num_proc=4  # Adjust based on your CPU cores
)

In [ ]:
output_path = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10_updated.dataset'

In [ ]:
dataset.save_to_disk(output_path, num_shards=32)

In [ ]:
import torch

def repeat_rel_y_coord(example):
    return {'rel_y_coord': example['rel_y_coord'].repeat_interleave(256, dim=0)}

dataset = dataset.map(repeat_rel_y_coord)

In [ ]:
dataset.save_to_disk(output_path, num_shards=32)

In [ ]:
# TODO: remove repeat interleave after it has been added to tokenization
dataset['rel_x_coord'] = dataset['rel_x_coord'].repeat_interleave(
    256, dim=1)

In [ ]:
dataset['rel_y_coord'] = dataset['rel_y_coord'].repeat_interleave(
    256, dim=1)

In [ ]:
len(rel_x_coord)

In [ ]:
test = torch.tensor(rel_x_coord)

In [ ]:
repeated_data = test.repeat_interleave(256, dim=1)

In [ ]:
rel_x_coord[0]

In [ ]:
z

In [ ]:
seg_tokens = dataset['seg_tokens']

In [ ]:
repeated_data = [val for val in rel_x_coord[0] for _ in range(3)]

In [ ]:
repeated_data

In [ ]:
len(rel_x_coord)

In [ ]:
rel_x_coord[-10:]

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('/nfs/team361/sb75/DATASETS/silver/xhs1000-39b_1p/adata_batch22.h5ad')

In [ ]:
adata

In [ ]:
from nichejepa.preprocessors.graph import construct_neighbor_graph

n_neighs = 10
radius = None
delaunay = None

print('Computing spatial neighborhood...')
# Construct neighbor graph
adata_neigh = construct_neighbor_graph(
    adata,
    n_neighs=n_neighs,
    radius=radius,
    delaunay=delaunay,
    include_self_loop=False,
    compute_neighbor_counts=False)

In [ ]:
adata_dict = {}

# Add coordinate tokens of index cells
adata_dict['rel_x_coord'] = [
    [0] for coord in adata.obsm['spatial'][:, 0].tolist()]
adata_dict['rel_y_coord'] = [
    [0] for coord in adata.obsm['spatial'][:, 1].tolist()]

# Add coordinate tokens of neighbor cells
for i in range(len(adata)):
    neighbor_indices = adata_neigh.obsp[
        'spatial_connectivities'][i].indices
    for j in neighbor_indices:
        adata_dict['rel_x_coord'][i].append((
            adata.obsm['spatial'][j, 0] - adata.obsm['spatial'][i, 0]).tolist())
        adata_dict['rel_y_coord'][i].append((
            adata.obsm['spatial'][j, 1] - adata.obsm['spatial'][i, 1]).tolist())

In [ ]:
(adata.obsm['spatial'][2, 0] - adata.obsm['spatial'][1, 0])

In [ ]:
adata.obsm['spatial'][2, 0]

In [ ]:
adata.obsm['spatial'][2, 0].astype(float)

In [ ]:
len(adata_dict['rel_x_coord'][200])

In [ ]:
adata_dict['rel_x_coord'][200]

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

# Get spatial coordinates and the connectivity graph
coords = adata_neigh.obsm['spatial']              # shape (n_cells, 2)
graph = adata_neigh.obsp['spatial_connectivities']  # scipy sparse matrix

# Ensure graph is in CSR format for efficient row access
graph = graph.tocsr()

# Store relative coordinates for each cell
relative_coords = []

for i in range(graph.shape[0]):
    # Find indices of neighbors
    neighbor_indices = graph[i].indices

    # Compute relative coordinates (neighbor - current)
    rel = coords[neighbor_indices] - coords[i]

    # Save result for cell i
    relative_coords.append(rel)

# Example: relative_coords[i] is an array of shape (num_neighbors_i, 2)

In [ ]:
relative_coords

In [ ]:
adata_neigh.obsp['spatial_connectivities']

In [ ]:
adata = sc.read_h5ad('/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/st_data/bronze/xhs1009-60b_1p/adata_skinpanel.h5ad')

In [ ]:
adata.obs['sample_id']

In [ ]:
adata.obs.columns

In [ ]:
adata.var

In [ ]:
adata2 = sc.read_h5ad("/nfs/team361/sb75/DATASETS/silver/mmb0-1b_smb1_1b_2p/starmap_plus_mouse_cns_batch15.h5ad")

In [ ]:
import re

dummy_indices = []

for value in token_dict.keys():
    match = re.match(r'spv_dummy_(\d+)', str(value))
    if match:
        dummy_indices.append(int(match.group(1)))

if dummy_indices:
    lowest_index = min(dummy_indices)
    print("Lowest dummy index:", lowest_index)
else:
    print("No dummy values found.")

In [ ]:
data_directory_path = '/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/st_data'

In [ ]:
dataset_name = 'xhs1009-17b_1p'

In [ ]:
lowest_index = 54

In [ ]:
for batch_idx, dummy_idx in enumerate(range(lowest_index, lowest_index + n_batches)):
    token_dict[f'spv_{dataset_name[3:7]}_batch{batch_idx}'] = token_dict.pop(f'spv_dummy_{dummy_idx}')

In [ ]:
adata_temp = sc.read_h5ad('/home/aih/sebastian.birk/workspace/projects/nichejepa-reproducibility/data/st_data/silver/xhs1009-17b_1p/adata_batch0.h5ad')

In [ ]:
len(data)

In [ ]:
data[-1]

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn.functional as F
import pytest
from nichejepa.utils.embedding import compute_running_mean_cosine_mult_occ  # Adjust import to your module path

def test_compute_running_mean_cosine_mult_occ():
    # Define small tensors for manual verification
    N, num_cell_genes, num_neb_genes, max_occ, D = 2, 2, 2, 2, 3

    # Set known embeddings
    cell_embs = torch.tensor([
        [[1., 0., 0.], [0., 1., 0.]],
        [[0., 1., 0.], [1., 0., 0.]]
    ])  # Shape: (2, 2, 3)

    cell_presence = torch.tensor([
        [1, 1],
        [1, 1]
    ])  # Shape: (2, 2)

    neb_occ = torch.tensor([
        [
            [[1., 0., 0.], [0., 1., 0.]],
            [[0., 1., 0.], [1., 0., 0.]]
        ],
        [
            [[1., 0., 0.], [0., 0., 0.]],
            [[0., 1., 0.], [0., 1., 0.]]
        ]
    ])  # Shape: (2, 2, 2, 3)

    neb_mask = torch.tensor([
        [[1, 1], [1, 1]],
        [[1, 0], [1, 1]]
    ])  # Shape: (2, 2, 2)

    total_cs, total_count = compute_running_mean_cosine_mult_occ(
        cell_embs, cell_presence, neb_occ, neb_mask
    )

    # Ensure the outputs have correct shapes
    assert total_cs.shape == (num_cell_genes, num_neb_genes)
    assert total_count.shape == (num_cell_genes, num_neb_genes)

    # Optionally print or assert exact values for known test case
    print("Total Cosine Similarity:\n", total_cs)
    print("Total Count:\n", total_count)

    print("Avg Cosine Similarity:\n", total_cs / total_count)

    # Example sanity check on a single element
    expected_count = 3  # For (0, 0): both samples count this, both have 2 occs
    assert total_count[0, 0].item() == expected_count


In [ ]:
test_compute_running_mean_cosine_mult_occ()

In [ ]:
test_compute_running_mean_cosine_mult_occ()

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p/adata_batch11.h5ad')

In [ ]:
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p/AT9-BRA-4-FO-2__filt_cells_filt_genes_raw_log1pnorm_counts.h5ad')

In [ ]:
adata.obs.columns

In [ ]:
adata.obs['leiden_log1pnorm_res-1']

In [ ]:
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(12, 8),
    facecolor='white',
    fontsize=10
)

sc.pl.spatial(
    adata,
    color='cell_type',
    spot_size=30)   

In [ ]:
batch_id = 1

adata = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/sb75/inigen/data/xenium_human_breast_cancer_batch{batch_id}.h5ad')

In [ ]:
adata.obs.head(1)

In [ ]:
adata.X.toarray()

In [ ]:
adata.obs.columns.tolist()

In [ ]:
adata.obs.head(1)

In [ ]:
batch_id = 1

adata = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/sb75/inflow/data/xenium_human_breast_cancer_batch{batch_id}.h5ad')

cell_type_key = 'cell_type'

# Dataset-specific metadata
adata.obs["batch"] = adata.obs["batch"].replace({"sample1": "Replicate 1", "sample2": "Replicate 2"})

trans_from=[['Epi_ABCC11+', 'Epi_FOXA1+', 'Epi_AGR3+', 'Epi_CENPF+', 'mgEpi_KRT14+', 'Epi_KRT14+'],['EC_CLEC14A+', 'EC_CAVIN2+'],['adipo_FB', 'GJB2+iKC-FB'],['EMT-Epi1_CEACAM6+', 'EMT-Epi2_CEACAM6+', 'EMT-Epi_SERPINA3+', 'EMT-Epi_KRT23+'],['DERL3+B', 'BANK1+B', 'B'],['eff_CD8+T1', 'eff_CD8+T2',],['tcm_CD4+T', 'CD161+FOXP3+T'],['NK/T'],['ADIPOQ+Mast'],['M2MØ', 'MMP12+miMØ'], ['DC1']]
trans_to = ['Epithelial', 'Endothelial', 'Fibroblast', 'EMT', 'B_cells', 'CD8+T', 'CD4+T', 'NK/T', 'Mast', 'MØ', 'DC']

adata.obs['cell_type'] = [str(i) for i in adata.obs['cell_states']]
for leiden, celltype in zip(trans_from, trans_to):
    for leiden_from in leiden:
        adata.obs['cell_type'][adata.obs['cell_type'] == leiden_from] = celltype

adata.obs[cell_type_key] = adata.obs[cell_type_key].replace("MØ", "Mɸ")
adata.obs["cell_states"] = adata.obs["cell_states"].replace("M2MØ", "M2Mɸ").replace("MMP12+miMØ", "MMP12+miMɸ")

adata.obs['sliceid'] = adata.obs['batch']

adata.write(f'/lustre/scratch126/cellgen/team361/sb75/inflow/data/xenium_human_breast_cancer_batch{batch_id}.h5ad')

In [ ]:
adata.obs.columns

In [ ]:
adata.obs.head(1)

In [ ]:
adata.write(f'/lustre/scratch126/cellgen/team361/sb75/inflow/data/xenium_human_breast_cancer_batch2.h5ad')

In [ ]:
import torch.nn as nn
import torch

In [ ]:
from typing import Optional, Tuple

import torch
import torch.nn as nn

from torch.nn.attention import SDPBackend, sdpa_kernel


class ValueEmbWeightsProjection(nn.Module):
    def __init__(self,
                 dim=100):
        """
        Project counts to value embedding weights.

        Parameters
        -----------
        dim:
            Dimensionality of the value embedding.        
        """
        super().__init__()
        self.linear1 = nn.Linear(1, dim)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(dim, dim)
        self.softmax = nn.Softmax(dim=-1)
    
    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        out = self.linear2(x)
        out = x + out # residual connection
        out = self.softmax(out)
        
        return out

In [ ]:
n_value_bins = 10
embed_dim = 3

n_special_values = 5
n_special_tokens = 3

In [ ]:
value_embed = nn.Embedding(
    n_value_bins,
    embed_dim)

In [ ]:
special_value_embed = nn.Embedding(
            2 + n_special_values + n_special_tokens,
            embed_dim,
            padding_idx=0)

In [ ]:
value_emb_weights_projection = ValueEmbWeightsProjection(
    dim=n_value_bins)

In [ ]:
counts = torch.tensor([[3., 3., 1., 0.]])

In [ ]:
value_emb_weights = value_emb_weights_projection(
    counts.unsqueeze(dim=-1))

In [ ]:
value_emb = torch.matmul(value_emb_weights, value_embed.weight)

In [ ]:
value_emb

In [ ]:
zero_counts_mask = counts == 0.0
zero_value_embed = special_value_embed(torch.tensor(0))
value_emb[zero_counts_mask] = zero_value_embed

In [ ]:
value_embed.weight

In [ ]:
value_emb.retain_grad()
loss = value_emb.sum()  # or any scalar function involving `output`

# Perform backpropagation
loss.backward()

In [ ]:
value_emb.grad

In [ ]:
value_emb.backward()

In [ ]:
import scanpy as sc
adata = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhe1008-1-20b_1p/section020_SOB26.h5ad')
adata

In [ ]:
raw_data_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver'
dataset_ids = 1008
obs_cols=None
uns_cols=['assay', 'tissue']

In [ ]:
adata_metadata = collect_adata_from_folder(
    raw_data_folder_path,
    all_cell_ids,
    dataset_ids,
    obs_cols,
    uns_cols)

In [ ]:
custom_palette_300 = {
    '0': '#F25454',
    '1': '#F2B354',
    '2': '#54F254',
    '3': '#54F2B3',
    '4': '#54B3F2',
    '5': '#B354F2',
    '6': '#F254B3',
    '7': '#F2B3B3',
    '8': '#B3F254',
    '9': '#54B3F2',
    '10': '#F25B54',
    '11': '#F2B854',
    '12': '#5BF254',
    '13': '#54F2B8',
    '14': '#54B8F2',
    '15': '#B854F2',
    '16': '#F254B8',
    '17': '#F2B8B8',
    '18': '#B8F254',
    '19': '#54B8F2',
    '20': '#F26154',
    '21': '#F2BD54',
    '22': '#61F254',
    '23': '#54F2BD',
    '24': '#54BDf2',
    '25': '#BD54F2',
    '26': '#F254BD',
    '27': '#F2BDBD',
    '28': '#BDF254',
    '29': '#54BDF2',
    '30': '#F26754',
    '31': '#F2C354',
    '32': '#67F254',
    '33': '#54F2C3',
    '34': '#54C3F2',
    '35': '#C354F2',
    '36': '#F254C3',
    '37': '#F2C3C3',
    '38': '#C3F254',
    '39': '#54C3F2',
    '40': '#F26D54',
    '41': '#F2C954',
    '42': '#6DF254',
    '43': '#54F2C9',
    '44': '#54C9F2',
    '45': '#C954F2',
    '46': '#F254C9',
    '47': '#F2C9C9',
    '48': '#C9F254',
    '49': '#54C9F2',
    '50': '#F27354',
    '51': '#F2CF54',
    '52': '#73F254',
    '53': '#54F2CF',
    '54': '#54CFF2',
    '55': '#CF54F2',
    '56': '#F254CF',
    '57': '#F2CFCF',
    '58': '#CFF254',
    '59': '#54CFF2',
    '60': '#F27954',
    '61': '#F2D554',
    '62': '#79F254',
    '63': '#54F2D5',
    '64': '#54D5F2',
    '65': '#D554F2',
    '66': '#F254D5',
    '67': '#F2D5D5',
    '68': '#D5F254',
    '69': '#54D5F2',
    '70': '#F27F54',
    '71': '#F2DB54',
    '72': '#7FF254',
    '73': '#54F2DB',
    '74': '#54DBF2',
    '75': '#DB54F2',
    '76': '#F254DB',
    '77': '#F2DBDB',
    '78': '#DBF254',
    '79': '#54DBF2',
    '80': '#F28554',
    '81': '#F2E354',
    '82': '#85F254',
    '83': '#54F2E3',
    '84': '#54E3F2',
    '85': '#E354F2',
    '86': '#F254E3',
    '87': '#F2E3E3',
    '88': '#E3F254',
    '89': '#54E3F2',
    '90': '#F28B54',
    '91': '#F2E954',
    '92': '#8BF254',
    '93': '#54F2E9',
    '94': '#54E9F2',
    '95': '#E954F2',
    '96': '#F254E9',
    '97': '#F2E9E9',
    '98': '#E9F254',
    '99': '#54E9F2',
    '100': '#F29154',
    '101': '#F2F054',
    '102': '#91F254',
    '103': '#54F2F0',
    '104': '#54F0F2',
    '105': '#F054F2',
    '106': '#F254F0',
    '107': '#F2F0F0',
    '108': '#F0F254',
    '109': '#54F0F2',
    '110': '#F29754',
    '111': '#F2F654',
    '112': '#97F254',
    '113': '#54F2F6',
    '114': '#54F6F2',
    '115': '#F654F2',
    '116': '#F254F6',
    '117': '#F2F6F6',
    '118': '#F6F254',
    '119': '#54F6F2',
    '120': '#F29D54',
    '121': '#F2FB54',
    '122': '#9DF254',
    '123': '#54F2FB',
    '124': '#54FBF2',
    '125': '#FB54F2',
    '126': '#F254FB',
    '127': '#F2FBFB',
    '128': '#FBF254',
    '129': '#54FBF2',
    '130': '#F2A354',
    '131': '#F30054',
    '132': '#A3F254',
    '133': '#54F300',
    '134': '#54F0F2',
    '135': '#0054F2',
    '136': '#F254F0',
    '137': '#F300F0',
    '138': '#F0F254',
    '139': '#54F0F2',
    '140': '#F2A954',
    '141': '#F30654',
    '142': '#A9F254',
    '143': '#54F306',
    '144': '#54F6F2',
    '145': '#0654F2',
    '146': '#F254F6',
    '147': '#F306F6',
    '148': '#F6F254',
    '149': '#54F6F2',
    '150': '#F2AF54',
    '151': '#F30C54',
    '152': '#AFF254',
    '153': '#54F30C',
    '154': '#54F2F2',
    '155': '#0C54F2',
    '156': '#F254F2',
    '157': '#F30CF2',
    '158': '#F2F254',
    '159': '#54F2F2',
    '160': '#F2B554',
    '161': '#F31254',
    '162': '#B5F254',
    '163': '#54F312',
    '164': '#54F2F2',
    '165': '#1254F2',
    '166': '#F254F2',
    '167': '#F312F2',
    '168': '#F2F254',
    '169': '#54F2F2',
    '170': '#F2BB54',
    '171': '#F31854',
    '172': '#BBF254',
    '173': '#54F318',
    '174': '#54F2F2',
    '175': '#1754F2',
    '176': '#F254F2',
    '177': '#F318F2',
    '178': '#F2F254',
    '179': '#54F2F2',
    '180': '#F2C154',
    '181': '#F31E54',
    '182': '#C1F254',
    '183': '#54F31E',
    '184': '#54F2F2',
    '185': '#1E54F2',
    '186': '#F254F2',
    '187': '#F31EF2',
    '188': '#F2F254',
    '189': '#54F2F2',
    '190': '#F2C754',
    '191': '#F32454',
    '192': '#C7F254',
    '193': '#54F324',
    '194': '#54F2F2',
    '195': '#2454F2',
    '196': '#F254F2',
    '197': '#F324F2',
    '198': '#F2F254',
    '199': '#54F2F2',
    '200': '#F2CD54',
    '201': '#F32A54',
    '202': '#CDF254',
    '203': '#54F32A',
    '204': '#54F2F2',
    '205': '#2A54F2',
    '206': '#F254F2',
    '207': '#F32AF2',
    '208': '#F2F254',
    '209': '#54F2F2',
    '210': '#F2D354',
    '211': '#F33054',
    '212': '#D3F254',
    '213': '#54F330',
    '214': '#54F2F2',
    '215': '#3054F2',
    '216': '#F254F2',
    '217': '#F330F2',
    '218': '#F2F254',
    '219': '#54F2F2',
    '220': '#F2D954',
    '221': '#F33654',
    '222': '#D9F254',
    '223': '#54F336',
    '224': '#54F2F2',
    '225': '#3654F2',
    '226': '#F254F2',
    '227': '#F336F2',
    '228': '#F2F254',
    '229': '#54F2F2',
    '230': '#F2DF54',
    '231': '#F33C54',
    '232': '#DFF254',
    '233': '#54F33C',
    '234': '#54F2F2',
    '235': '#3C54F2',
    '236': '#F254F2',
    '237': '#F33CF2',
    '238': '#F2F254',
    '239': '#54F2F2',
    '240': '#F2E554',
    '241': '#F34254',
    '242': '#E5F254',
    '243': '#54F342',
    '244': '#54F2F2',
    '245': '#4254F2',
    '246': '#F254F2',
    '247': '#F342F2',
    '248': '#F2F254',
    '249': '#54F2F2',
    '250': '#F2EB54',
    '251': '#F34854',
    '252': '#EBF254',
    '253': '#54F348',
    '254': '#54F2F2',
    '255': '#4854F2',
    '256': '#F254F2',
    '257': '#F348F2',
    '258': '#F2F254',
    '259': '#54F2F2',
    '260': '#F2F154',
    '261': '#F34E54',
    '262': '#F1F254',
    '263': '#54F34E',
    '264': '#54F2F2',
    '265': '#4E54F2',
    '266': '#F254F2',
    '267': '#F34EF2',
    '268': '#F2F254',
    '269': '#54F2F2',
    '270': '#F2F754',
    '271': '#F35454',
    '272': '#F7F254',
    '273': '#54F354',
    '274': '#54F2F2',
    '275': '#5454F2',
    '276': '#F254F2',
    '277': '#F354F2',
    '278': '#F2F254',
    '279': '#54F2F2',
    '280': '#F2FF54',
    '281': '#F35A54',
    '282': '#FFF254',
    '283': '#54F35A',
    '284': '#54F2F2',
    '285': '#5A54F2',
    '286': '#F254F2',
    '287': '#F35AF2',
    '288': '#F2F254',
    '289': '#54F2F2',
    '290': '#F20554',
    '291': '#F35F54',
    '292': '#05F254',
    '293': '#54F35F',
    '294': '#54F2F2',
    '295': '#5F54F2',
    '296': '#F254F2',
    '297': '#F35FF2',
    '298': '#F2F254',
    '299': '#54F2F2',
    "-1" : "#E1D9D1",
    "None" : "#E1D9D1"
}

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt
import scanpy as sc

In [ ]:
from analysis_utils import

In [ ]:
artifacts_folder_path = '/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts'

In [ ]:
load_timestamp = '19022025_023508_679'
corpus_size = 72000000
step_size = 2000000
read_checkpoint = 'ep0'

In [ ]:
emb_key = 'neighborhood_emb_layer_6'

In [ ]:
steps = int(corpus_size / step_size)

In [ ]:
adata_list = []

for i in range(steps):
    try:
        adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/adata_{read_checkpoint}_{step_size}_{i}.h5ad')
        if emb_key == 'cell_emb_layer_6':
            del adata.obsm['neighborhood_emb_layer_6']
        elif emb_key == 'neighborhood_emb_layer_6':
            del adata.obsm['cell_emb_layer_6']
        adata_list.append(adata)
        del adata
    except:
        print(f'Could not find file with file name adata_{read_checkpoint}_{step_size}_{i}.h5ad.')

adata_full = sc.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata_full

In [ ]:
!free -h

In [ ]:
adata_full.obs.head(1)

In [ ]:
random_state = 42
sample_frac = 0.2

In [ ]:
sample_frac = 0.2
random_state = 42

In [ ]:
adata_full.obs[['dataset', 'batch', 'cell']] = [s.split('_') for s in adata_full.obs.index]
adata_full.obs['dataset_batch'] = adata_full.obs['dataset'].astype(str) + adata_full.obs['batch'].astype(str)
adata_full.obs['cell'] = adata_full.obs['cell'].astype(int)

# Stratified sampling: sample 20% from each batch
sampled_df = adata_full.obs.groupby('dataset_batch', group_keys=False).apply(
    lambda x: x.sample(frac=sample_frac, replace=False, random_state=random_state))

In [ ]:
len(set(sampled_df.index.tolist()))

In [ ]:
adata_temp = adata_full[sampled_df.index]

In [ ]:
adata_full = adata_temp

In [ ]:
del(adata_full)

In [ ]:
emb_key = 'neighborhood_emb_layer_6'

In [ ]:
sc.pp.neighbors(adata_full,
                n_neighbors=15,
                use_rep=emb_key,
                key_added='neighborhood') 

In [ ]:
sc.tl.umap(adata_full,
           neighbors_key='neighborhood')

In [ ]:
adata_full.obs['dataset_batch'].unique()

In [ ]:
import random

random.seed(42)

custom_palette_300 = {str(i): '#{:06X}'.format(random.randint(0, 0xFFFFFF)) for i in range(300)}
print(custom_palette_300)

In [ ]:
dataset_batch_palette = {new_key: value for new_key, value in zip(
    adata_full.obs['dataset_batch'].unique().tolist(),
    custom_palette_300.values())}

In [ ]:
dataset_batch_palette

In [ ]:
labels = ['assay', 'tissue', 'dataset_batch', 'gene_panel_size']
# Plot (aggregated) neighborhood gene embeddings, colored by labels
for label in labels:
    if label == 'assay':
        palette = 'Dark2'
    elif label == 'tissue':
        palette = 'tab20'
    elif label == 'dataset_batch':
        palette = dataset_batch_palette
    sc.pl.umap(adata_full,
               color=label,
               palette=palette,
               show=False)
    plt.savefig(f'{artifacts_folder_path}/hst_corpus_70m/{load_timestamp}/hst_corpus_70m_{sample_frac}_subsample_{label}_umap.svg',
                dpi=300,
                bbox_inches='tight')

In [ ]:
artifacts_folder_path

In [ ]:
!free -h

In [ ]:
len(adata_temp)

In [ ]:
len(set(sampled_df.index.tolist()))

In [ ]:
sampled_df['batch'].unique()

In [ ]:
len(sampled_df)

In [ ]:
sampled_df.head(5)

In [ ]:
adata_full.obs.index.tolist()

In [ ]:
import time
from datetime import timedelta

start_time = time.time()
print("Start time:", start_time)

# Compute neighbors and UMAP 
sc.pp.neighbors(adata_full,
                n_neighbors=15,
                use_rep=emb_key,
                key_added='neighborhood')

intermediate_time = time.time()
print("Intermediate time:", intermediate_time)

elapsed_time = intermediate_time - start_time
print("Elapsed time:", str(timedelta(seconds=elapsed_time)))

sc.tl.umap(adata,
           neighbors_key='neighborhood')

end_time = time.time()
print("End time:", end_time)

elapsed_time = end_time - start_time
print("Elapsed time:", str(timedelta(seconds=elapsed_time)))

In [ ]:
adata.write('./temp.h5ad')

In [ ]:
!free -h

In [ ]:
adata.obs.columns

In [ ]:
adata.uns

In [ ]:
from pathlib import Path
folder_path = Path(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p')
adata_path_list = [p for p in folder_path.rglob("*")]

In [ ]:
for adata_path in adata_path_list:
    adata = sc.read_h5ad(adata_path)
    print(f'{adata_path}: {adata.uns["batch"]}')

In [ ]:
import pickle
import pprint

In [ ]:
token_dict_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl'

with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

In [ ]:
pp = pprint.PrettyPrinter(indent=4, width=40, depth=2)
pp.pprint(token_dict)

In [ ]:
token_dict['spv_gene_panel4949']

In [ ]:
token_dict['spv_embryo'] = token_dict.pop('dummy_0')

token_dict['spv_gene_panel4949'] = token_dict.pop('dummy_1')

for i in range(2,22):
    token_dict[f'spv_1008_batch{i-2}'] = token_dict.pop(f'dummy_{i}')

In [ ]:
pp = pprint.PrettyPrinter(indent=4, width=40, depth=2)
pp.pprint(token_dict)

In [ ]:
import scanpy as sc

In [ ]:
dataset_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS'

In [ ]:
adata = sc.read_h5ad(f'{dataset_folder_path}/silver/xhe1008-1-20b_1p/section173_SOB26.h5ad')

In [ ]:
adata

In [ ]:
adata.var

In [ ]:
# coronal 10 slide_id
# sagittal 6
# 

In [ ]:
adata

In [ ]:
adata.obs.head(1)

In [ ]:
adata.obs['slide_id'].unique()

In [ ]:
adata_batch = adata[adata.obs['slide_id'] == 'B2']

In [ ]:
sc.pl.spatial(adata_batch,
              color='subregion',
              spot_size=30)

In [ ]:
with open(token_dict_path, 'wb') as f:
    pickle.dump(token_dict, f)

In [ ]:
import json

In [ ]:
json_file = '/lustre/scratch126/cellgen/team361/data/sanger_whole_embryo/xenium/section020_SOB26/output-XETG00155__0004457__SOB26-WEM-FFPE-1-S20__20240912__123924/gene_panel.json'

In [ ]:
with open(json_file, 'r') as file:
    gene_panel = json.load(file)

In [ ]:
gene_panel.keys()

In [ ]:
len(gene_panel['payload']['targets'])

In [ ]:
gene_panel['metadata']

In [ ]:
# Pickle the list
with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    train = pickle.load(f)

with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    test = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    new_train = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    new_test = pickle.load(f)

print(len(train))
print(len(test))
print(len(new_train))
print(len(new_test))

train_matches = set(train).intersection(new_train)
print(len(train_matches))
test_matches = set(test).intersection(new_test)
print(len(test_matches))

train_differences = []
for i in range(len(train)):
    if train[i] != new_train[i]:
        train_differences.append((i, train[i], new_train[i]))
print(train_differences)
print(len(differences))

test_differences = []
for i in range(len(test)):
    if test[i] != new_test[i]:
        test_differences.append((i, test[i], new_test[i]))
print(test_differences[-10:])

In [ ]:
import scanpy as sc
adata = sc.read_h5ad(/adata_batch0.h5ad')

In [ ]:
import os

In [ ]:
for subdir, _, files in os.walk('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(file_path)
            print(adata.obs['cell_id'].values[0])
            #print(adata.obs['sample'].unique())
            print(adata.uns['batch'])
            print("================")

In [ ]:
adata.X.toarray()

In [ ]:
import numpy as np
adata.X = adata.X.astype(np.float64)

In [ ]:
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color='niche_name')

In [ ]:
scales_counts = sc.pp.normalize_total(adata, target_sum=None, inplace=False)
adata.X = sc.pp.log1p(scales_counts["X"], copy=True)
sc.pp.pca(adata, svd_solver="arpack", use_highly_variable=False)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color='niche_name')

In [ ]:
sc.pl.spatial(adata,
              color='niche_name',
              spot_size=30)

In [ ]:
adata.obs['niche_name'].value_counts()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    inplace=True
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data=adata.obs, x='n_genes_by_counts', bins=100)
plt.xlabel('Number of expressed genes per cell')
plt.ylabel('Number of cells')
plt.show()

In [ ]:
adata.obsm['spatial_connectivities']

In [ ]:
# Load hold out skin samples and concatenate in one adata object
adata_list = []

for subdir, _, files in os.walk('/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad'):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            print(adata.X)
            adata.X = adata.X.astype(int64)
            print(adata.X)

adata = ad.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata2 = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/silver/mmb0-1b_smb1-1b_1p/harmonised_starmap_plus_mouse_cns_batch15_shared_genes.h5ad')

In [ ]:
import pickle

# Path to your pickle file
pickle_file_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-graph-tokenizer/human_cohort1_10m/token_dictionary.pkl'

# Load the pickle file
with open(pickle_file_path, 'rb') as file:
    token_dict = pickle.load(file)

# Now `token_dict` is a Python dictionary
print(token_dict)

In [ ]:
spv_count = sum(1 for key in token_dict if "spv" in key)
spv_count

In [ ]:
len(token_dict)

In [ ]:
import os
import pickle

import scanpy as sc
import squidpy as sq

In [ ]:
folder_path = '/lustre/scratch126/cellgen/team361/DATASETS/silver/xhs1000-39b_1p'
token_dict_path = '/lustre/scratch126/cellgen/team361/DATASETS/tokenizer/cell-neighborhood-tokenizer/xhs1000-39b_1p/token_dictionary.pkl'

In [ ]:
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

In [ ]:
len(token_dict)

In [ ]:
count_spv_keys = sum(1 for key in token_dict if key.startswith("spv"))
count_spv_keys

In [ ]:
token_dict[list(token_dict.keys())[-1:][0]]

In [ ]:
token_dict['ENSG00000292373']

In [ ]:
max_nonzero_counts = 0  # Initialize a variable to store the maximum count
neighbor_max_nonzero_counts = 0

# Iterate through all .h5ad files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.h5ad'):
        file_path = os.path.join(folder_path, filename)
        
        # Load the .h5ad file
        adata = sc.read_h5ad(file_path)

        sq.gr.spatial_neighbors(adata,
                                coord_type='generic',
                                spatial_key='spatial',
                                delaunay=True,
                                set_diag=True)

        adata.layers['X_neighborhood'] = (
                adata.obsp['spatial_connectivities'].T @ adata.X)
        
        # Calculate nonzero counts per cell
        nonzero_counts = (adata.X > 0).sum(axis=1)
        neighbor_nonzero_counts = (adata.layers['X_neighborhood'] > 0).sum(axis=1)
        
        # Get the maximum nonzero count in the current file
        file_max_nonzero = nonzero_counts.max()
        file_max_neighbor_nonzero = neighbor_nonzero_counts.max()
        
        # Update the overall maximum if the current file has a higher value
        if file_max_nonzero > max_nonzero_counts:
            max_nonzero_counts = file_max_nonzero

        if file_max_neighbor_nonzero > neighbor_max_nonzero_counts:
            neighbor_max_nonzero_counts = file_max_neighbor_nonzero

In [ ]:
adata.obs

In [ ]:
max_nonzero_counts

In [ ]:
neighbor_max_nonzero_counts

In [ ]:
adata_test = sc.read_h5ad('/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhs1000-1b_1p/adata_5k_xenium_skin.h5ad')

In [ ]:
adata_test.obs['info_id']

In [ ]:
batch_mapping_dict = {value: idx for idx, value in enumerate(adata_test.obs['info_id'].unique().tolist())}

In [ ]:
batch_mapping_dict # 19, 11, 32

In [ ]:
2+3

In [ ]:
import os
import shutil

main_directory = '/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhe1010-20b_1p'
keyword = 'output'  # Adjust this to match your criteria

for subdir in os.listdir(main_directory):
    subdir_path = os.path.join(main_directory, subdir)
    if os.path.isdir(subdir_path):
        # Find nested directories in the subdirectory.
        nested_dirs = [
            d for d in os.listdir(subdir_path)
            if os.path.isdir(os.path.join(subdir_path, d))
        ]
        
        # Option A: If there is exactly one nested directory, zip that one.
        if len(nested_dirs) == 1:
            target = nested_dirs[0]
        # Option B: If there are multiple, pick the one whose name contains the keyword.
        else:
            target = next((d for d in nested_dirs if keyword in d), None)
        
        if target is not None:
            target_dir_path = os.path.join(subdir_path, target)
            archive_base = os.path.join(subdir_path, target)
            shutil.make_archive(base_name=archive_base, format='zip', root_dir=target_dir_path)
            print(f"Created zip archive: {archive_base}.zip")
        else:
            print(f"No matching nested directory found in {subdir_path}")

In [ ]:
'/lustre/scratch126/cellgen/team361/DATASETS/bronze/xhe1009-20b_1p/section056_SOB26/output-XETG00155__0003607__SOB25-WEM-FFPE-1-S56__20240912__123924.zip'

In [ ]:
chmod -R u+rw /lustre/scratch126/cellgen/team361/DATASETS/bronze/xhe1009-20b_1p


In [ ]:
import os
import torch

import logging
import torch.distributed as dist
from app.train import train
from nichejepa.datasets.utils import prepare_dataset
from nichejepa.utils.config import create_params_from_YAML_wandb_config
import wandb
import sys
# Add the root directory to sys.path
#sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "../..")))
import argparse
import time
import importlib
import warnings
from datetime import datetime
from datetime import timedelta
import random
from socket import gethostname

warnings.filterwarnings("ignore")
# logger
logging.basicConfig()
logger = logging.getLogger()

# ==========================

# Function to retrieve and log distributed environment variables
def get_distributed_info():
    """
    Retrieves distributed training environment variables and logs them.

    Returns:
        tuple: (WORLD_RANK, LOCAL_RANK, WORLD_SIZE)
    """
    if "SLURM_PROCID" in os.environ:
        WORLD_RANK = int(os.environ["SLURM_PROCID"])
        WORLD_SIZE = int(os.environ["WORLD_SIZE"])
        GPUS_PER_NODE = int(os.environ["SLURM_GPUS_ON_NODE"])
        assert GPUS_PER_NODE == torch.cuda.device_count()
        print(f"Hello from rank {WORLD_RANK} of {WORLD_SIZE} on {gethostname()} where there are" \
            f" {GPUS_PER_NODE} allocated GPUs per node.", flush=True)
        LOCAL_RANK = WORLD_RANK - GPUS_PER_NODE * (WORLD_RANK // GPUS_PER_NODE)
    if "LOCAL_RANK" in os.environ:
        # Environment variables set by torch.distributed.launch or
        # torchrun
        LOCAL_RANK = int(os.environ["LOCAL_RANK"])
        WORLD_SIZE = int(os.environ["WORLD_SIZE"])
        WORLD_RANK = int(os.environ["RANK"])
    elif "OMPI_COMM_WORLD_LOCAL_RANK" in os.environ:
        # Environment variables set by mpirun
        LOCAL_RANK = int(os.environ["OMPI_COMM_WORLD_LOCAL_RANK"])
        WORLD_SIZE = int(os.environ["OMPI_COMM_WORLD_SIZE"])
        WORLD_RANK = int(os.environ["OMPI_COMM_WORLD_RANK"])
    else:
        import sys
        sys.exit("Can't find the environment variables for local rank")

    # Print the ranks
    print(f"World rank: {WORLD_RANK}, Local rank: {LOCAL_RANK}, World size: {WORLD_SIZE}")

    return WORLD_RANK, LOCAL_RANK, WORLD_SIZE


def setup_for_distributed(is_master):
    """
    This function disables printing when not in master process
    """
    import builtins as __builtin__
    builtin_print = __builtin__.print

    def print(*args, **kwargs):
        force = kwargs.pop('force', False)
        if is_master or force:
            builtin_print(*args, **kwargs)

    __builtin__.print = print

In [ ]:
import os

os.environ["LOCAL_RANK"] = '0'
os.environ["WORLD_SIZE"] = '1'
os.environ["RANK"] = '0'
os.environ["EXPERIMENT_NAME"] = 'nichejepa_finetuning'
os.environ["RUN_NAME"] = 'xhk1007-CV1-41b_1p'
os.environ['MASTER_ADDR'] = 'farm22-gpu0301'
os.environ['MASTER_PORT'] = '12345'

In [ ]:
# Retrieve distributed environment variables
WORLD_RANK, LOCAL_RANK, WORLD_SIZE = get_distributed_info()

"""
# Parse arguments
parser = argparse.ArgumentParser(
    formatter_class=argparse.ArgumentDefaultsHelpFormatter)
parser.add_argument(
    '--backend',
    type=str,
    help='Backend for distributed training.',
    default='nccl',
    choices=['nccl', 'gloo', 'mpi'],)
parser.add_argument(
    '--fname', 
    type=str, 
    default='configs.yaml',
    help='Name of the config file to load.',)
"""
backend = 'nccl'
fname = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/29062025_202056_303/params.yaml'
run_timestamp = '29062025_202056_303'

#args = parser.parse_args()
# backend = args.backend
experiment_name = os.environ.get('EXPERIMENT_NAME')
run_name = os.environ.get('RUN_NAME')
run_id = f'{experiment_name}_{run_name}'

# Retrieve params
params = create_params_from_YAML_wandb_config(
    fname, # args.fname,
    logger)
#logger.info(f'Called with params from {args.fname}.')
logger.info(f'Called with params from {fname}.')
logger.info(f'Params: {params}.')

# Init logging and create folders
if WORLD_RANK==0:
    wandb.init(
        project='nichejepa-finetuning',
        id=run_id,
        resume="allow",
        group="multi_node_training",
        mode='online')
    artifact_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
    print(f'Run timestamp: {run_timestamp}.')
    if not wandb.run.resumed:
        wandb.config.run_timestamp = run_timestamp
    if params['state']['folder_path'] is None:
        folder_path = os.path.join(artifact_folder_path,
                                   params['data']['dataset_name'],
                                   run_timestamp)
    else:
        folder_path = params['state']['folder_path']
else:
    folder_path = None

In [ ]:
print(f"tcp://{os.environ['MASTER_ADDR']}:{os.environ['MASTER_PORT']}")

torch.cuda.set_device(LOCAL_RANK)

In [ ]:
# Initialize the distributed backend
dist.init_process_group(
    backend=backend,
    init_method=f"tcp://{os.environ['MASTER_ADDR']}:{os.environ['MASTER_PORT']}",
    rank=WORLD_RANK,
    world_size=WORLD_SIZE,
)

In [ ]:
dist.barrier()
setup_for_distributed(LOCAL_RANK == 0)

In [ ]:
split = dataset.train_test_split(test_size=0.2, seed=0)

train_dataset = split['train']
test_dataset = split['test']

fine_tune(params,
          train_dataset,
          test_dataset,
          save_folder_path=folder_path,
          use_peft=False,
          LOCAL_RANK=LOCAL_RANK,
          WORLD_RANK=WORLD_RANK)

adata = predict(
    model_folder_path,
    test_dataset)

In [ ]:
# ~4sec (not keep in memory)
import datetime
from datasets import load_from_disk
print(f"Start: {datetime.datetime.now()}")
ds = load_from_disk(
    '/nfs/team361/sb75/DATASETS/gold/cell-graph-tokenizer/hst_corpus_80m/hst_corpus_80m_32_gene_corrected_read_depth_nonzero_mean_None_None_None_shifted_log_knn_10.dataset')
print(f"End: {datetime.datetime.now()}")

train_mode = True

fields_to_remove = ['cell_degrees', # TODO needed for old tokenized data
                    'batch_value_token',
                    'gene_panel_value_token',
                    'assay_value_token',
                    'species_value_token',
                    'tissue_value_token',
                    'cls_tokens',
                    'cell_total_counts',
                    'cell_n_probed_genes']
for special_token in ['batch', 'gene_panel', 'assay', 'species', 'tissue']:
    if special_token not in params['meta']['special_tokens']:
        fields_to_remove += [f'{special_token}_token']
        fields_to_remove += [f'{special_token}_value']
if 'cell_pos_enc' not in params['meta'].keys():
    fields_to_remove += ['rel_x_coord']
    fields_to_remove += ['rel_y_coord']
elif params['meta']['cell_pos_enc'] == 'segment':
    fields_to_remove += ['rel_x_coord']
    fields_to_remove += ['rel_y_coord']
if train_mode:
    fields_to_remove += [f'cell_id']

existing_fields = [
    col for col in fields_to_remove if col in ds.column_names]

if existing_fields:
    ds = ds.remove_columns(existing_fields)

columns = list(ds.features.keys())
if not train_mode:
    columns.remove("cell_id")
ds.set_format(
    type="torch",
    columns=columns,
    output_all_columns=True)

# ~1sec
print(f"Start: {datetime.datetime.now()}")
item = ds[0]

print(f"End: {datetime.datetime.now()}")